In [ ]:
pip install sunpy[all]

In [1]:
import pandas as pd
from sunpy.net import attrs as a
from sunpy.net.hek import HEKClient
from google.colab import files

hek = HEKClient()

# Define smaller time windows to bypass HEK's 100-result per query cap
time_windows = [
    ("2024-05-01", "2024-05-07"),
    ("2024-05-07", "2024-05-14"),
    ("2024-05-14", "2024-05-20"),
]

all_dfs = []

for start, end in time_windows:
    response = hek.search(
        a.Time(start, end),
        a.hek.EventType("FL"),
        a.hek.AR.NOAANum == 13664,
    )
    if len(response) > 0:
        # Keep 1D columns to prevent multidimensional array conversion issues
        cols_1d = [col for col in response.colnames if len(response[col].shape) <= 1]
        df_chunk = response[cols_1d].to_pandas()
        all_dfs.append(df_chunk)

# Combine and deduplicate entries
df_all = pd.concat(all_dfs, ignore_index=True)
df_all = df_all.drop_duplicates(subset=["event_starttime", "event_peaktime"]).reset_index(drop=True)

# Select key metadata columns
cols = [
    "event_starttime",
    "event_peaktime",
    "event_endtime",
    "fl_goescls",
    "ar_noaanum",
    "hpc_x",
    "hpc_y",
    "frm_name",
]

valid_cols = [c for c in cols if c in df_all.columns]
df_final = df_all[valid_cols]

# Filter strictly for AR 13664 and sort chronologically
df_final = df_final[df_final["ar_noaanum"] == 13664].sort_values(by="event_starttime").reset_index(drop=True)

print(f"Total flares retrieved for AR 13664: {len(df_final)}")

# Save to CSV and prompt browser download
csv_filename = "AR13664_flares.csv"
df_final.to_csv(csv_filename, index=False)
files.download(csv_filename)

: 